# Chapter 2 — Non-Parametric Approaches

教材 pp. 18–34；PDF 页序为教材页码加 11。

历史重估前的风险因子变动、定价和数据清洗由调用层准备。本章计算器直接接收损失样本或标准化所需列。

下列未明确标为教材例题的数据均为合成验证数据，不声称复现教材中的原始市场样本。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from historical_calculator import *

经验 ES 的部分概率与教材尾部平均；pp.19–24。

In [2]:
data = pd.DataFrame({'loss':[1.,2.,3.,4.]})
model = HistoricalDistribution(data,'loss')
assert model.quantile(.75) == 3
assert_allclose(model.expected_shortfall(.625), 11/3)
weighted = age_weights(data,.95)
assert_allclose(weighted.weight.sum(),1)
assert weighted.weight.is_monotonic_increasing
print(model.expected_shortfall(.75))

4.0


KDE 对称性、波动率缩放和可重复 bootstrap；pp.19–20,25,31–34。

In [3]:
data = pd.DataFrame({'loss':[-2.,-1.,0.,1.,2.], 'vol':.2})
assert_allclose(KernelDistribution(data,'loss').quantile(.5,(-5,5)),0,atol=1e-8)
assert_allclose(volatility_scale(data,'loss','vol',.4).scaled,2*data.loss)
a = bootstrap_estimates(data,lambda x:x.loss.mean(),30,np.random.default_rng(7),2)
b = bootstrap_estimates(data,lambda x:x.loss.mean(),30,np.random.default_rng(7),2)
assert_allclose(a,b)
print(quantile_interval(data,'loss',.5))

lower   -inf
upper    inf
dtype: float64


## AGARCH、相关调整与KDE尾积分；pp.20,26–27 / PDF31,37–38

In [4]:
# Example 2.1：旧相关0.3，新相关0.9，对应变换矩阵下行约[0.7629,0.4569]。
basis=pd.DataFrame(np.eye(2),columns=['a','b'])
adjusted=correlation_scale(basis,['a','b'],pd.DataFrame([[1,.3],[.3,1]]),pd.DataFrame([[1,.9],[.9,1]]))
assert_allclose(adjusted.to_numpy().T,[[1,0],[.7629,.4569]],atol=.0001)
model=AsymmetricGARCH(.01,.1,.8,.02,0)
data=pd.DataFrame({'r':[.1,-.1,.2]})
filtered=model.filter(data,'r',.04)
assert_allclose(filtered.variance.iloc[1],.01+.1*(.1+.02)**2+.8*.04)
paths=model.filtered_paths(filtered,'standardized_residual',.04,5,100,np.random.default_rng(42))
assert paths.shape==(5,100)
# 对称KDE的中位数为0；E[X|X>0]可用高斯混合的解析绝对矩一半核对。
from scipy.stats import norm
data=pd.DataFrame({'x':[-1.,0.,1.]})
kde=KernelDistribution(data,'x')
sigma=np.sqrt(kde.kde.covariance[0,0])
mu=data.x.to_numpy()
absolute=(sigma*np.sqrt(2/np.pi)*np.exp(-mu**2/(2*sigma**2))+mu*(2*norm.cdf(mu/sigma)-1)).mean()
assert_allclose(kde.expected_shortfall(.5,(-5,5)),absolute,rtol=1e-8)